# Exercise 2026/10/01

## 題目：十二生肖

撰寫 Python 程式，讓使用者輸入年份（可能是西元或民國），找尋其相對應的十二生肖。

十二生肖以十二年為一週期：猴（monkey）、雞（rooster）、狗（dog）、豬（pig）、鼠（rat）、牛（ox）、虎（tiger）、兔（rabbit）、龍（dragon）、蛇（snake）、馬（horse）、羊（sheep）。

In [ ]:
# 十二生肖程式
# 民國年 = 西元年 - 1911，因此民國年可以先轉成西元年。

zodiac = [
    "monkey", "rooster", "dog", "pig", "rat", "ox",
    "tiger", "rabbit", "dragon", "snake", "horse", "sheep"
]

year = int(input("請輸入年份（西元或民國）："))

# 判斷輸入的是民國年還是西元年。
# 一般民國年份小於 1912，因此此作業可用 1912 作為簡單判斷界線。
if year < 1912:
    ad_year = year + 1911
else:
    ad_year = year

# 2004 年為猴年，因此用 (年份 - 2004) % 12 找生肖位置。
animal = zodiac[(ad_year - 2004) % 12]

print(f"{year} 年對應的西元年份為 {ad_year} 年，十二生肖是 {animal}。")

## 浮點數問題

### 1. 0.1 的二進位表示法為何？

十進位的 `0.1` 在二進位中不是有限位數，而是一個循環小數。

使用「小數乘 2、取整數部分」的方法：

| 步驟 | 小數 × 2 | 整數部分 | 新的小數部分 |
|---|---:|---:|---:|
| 1 | 0.1 × 2 = 0.2 | 0 | 0.2 |
| 2 | 0.2 × 2 = 0.4 | 0 | 0.4 |
| 3 | 0.4 × 2 = 0.8 | 0 | 0.8 |
| 4 | 0.8 × 2 = 1.6 | 1 | 0.6 |
| 5 | 0.6 × 2 = 1.2 | 1 | 0.2 |
| 6 | 0.2 × 2 = 0.4 | 0 | 0.4 |
| 7 | 0.4 × 2 = 0.8 | 0 | 0.8 |
| 8 | 0.8 × 2 = 1.6 | 1 | 0.6 |

從第 2 步之後會再次出現 `0.2 → 0.4 → 0.8 → 0.6` 的循環，所以：

**`0.1₁₀ = 0.00011001100110011...₂`**

也可以寫成 `0.000(1100)₂`，其中 `1100` 不斷重複。

**看到的現象：** 0.1 在十進位是有限小數，但轉成二進位後卻變成無限循環小數。因此電腦使用有限的 bit 儲存時，無法精確儲存數學上的 0.1，只能儲存非常接近 0.1 的值。

### 2. 如果是無限多位，Python 如何解決？

Python 的一般 `float` 採用 **IEEE 754 double-precision（雙精度）** 浮點數格式，使用有限的 64 bits 儲存數值：

- 1 bit：sign（正負號）
- 11 bits：exponent（指數）
- 52 bits：fraction / significand fraction（有效數字部分）

由於 `0.1` 的二進位表示是無限循環小數，Python 不可能把全部位數存下來，因此會截取有限位數，再選擇最接近的 IEEE 754 可表示值。Python 浮點數使用的舍入方式可概括為 **round to nearest, ties to even（最近值；剛好在中間時採四捨六入五成雙）**。

因此，Python 儲存的 `0.1` 並不是數學上完全精確的 0.1，而是最接近 0.1 的 double-precision 浮點數。這也是之後 `0.1 + 0.2` 可能出現 `0.30000000000000004` 的根本原因之一。

### 3. Python 如何知道浮點數的精度？

執行：

```python
import sys
print(sys.float_info)
```

在一般使用 IEEE 754 binary64 的 Python 環境中，會看到類似：

```text
sys.float_info(max=1.7976931348623157e+308, max_exp=1024,
max_10_exp=308, min=2.2250738585072014e-308,
min_exp=-1022, min_10_exp=-307, dig=15,
mant_dig=53, epsilon=2.220446049250313e-16,
radix=2, rounds=1)
```

主要欄位意思如下：

| 欄位 | 意義 |
|---|---|
`max` | 可表示的最大有限浮點數 |
`min` | 正規化浮點數中最小的正數 |
`dig` | 保證至少約有 15 位十進位有效數字可以保持正確 |
`mant_dig` | 有效數字的二進位位數；double 通常是 53 bits |
`epsilon` | 1.0 與下一個可表示浮點數之間的差，約為 `2.22 × 10⁻¹⁶` |
`radix` | 浮點數使用的基數，通常為 2 |
`rounds` | 浮點運算的舍入方式；`1` 代表 round to nearest |

因此 Python 是透過底層浮點數格式與系統提供的浮點數資訊，知道 `float` 能表示的範圍與精度。

In [ ]:
import sys
print(sys.float_info)

### 4. 為什麼 `0.1 + 0.2` 不是 0.3？怎麼解決？

執行：

```python
print(0.1 + 0.2)
```

通常得到：

```text
0.30000000000000004
```

原因是 `0.1` 和 `0.2` 都不能用有限的二進位浮點數精確表示。它們先各自被近似儲存，做加法後結果再經過浮點數表示與顯示，因此會看到非常小的誤差。

**方法一：比較浮點數時使用容許誤差**

```python
import math

print(math.isclose(0.1 + 0.2, 0.3))
```

結果會是：

```text
True
```

如果需要自己設定誤差，可以使用：

```python
abs((0.1 + 0.2) - 0.3) < 1e-10
```

**方法二：需要十進位精確運算時使用 `decimal.Decimal`**

```python
from decimal import Decimal

a = Decimal("0.1")
b = Decimal("0.2")
print(a + b)
```

結果為：

```text
0.3
```

注意：建立 `Decimal` 時最好從字串建立，例如 `Decimal("0.1")`，不要先用 `float` 再轉換。

### 5. 這個方法對一般兩個浮點數加法，在科學或金融上是好方法嗎？

要看使用情境，**不能把 `round()` 或固定小數位數當成所有浮點數問題的通用解法**。

#### 科學計算

科學計算通常可以接受一定程度的浮點誤差，因此一般會使用 IEEE 754 `float`，並根據問題的尺度使用適當的容許誤差，例如 `math.isclose()`。直接把每一次運算都 `round()` 可能累積額外的誤差，甚至改變計算結果。

例如：

```python
import math

math.isclose(a, b, rel_tol=1e-9, abs_tol=0.0)
```

其中 `rel_tol` 與 `abs_tol` 應依實際科學問題設定，而不是永遠使用同一個數字。

#### 金融計算

金融資料常要求十進位小數具有明確且可預期的意義，例如金額精確到分。此時通常比起一般 binary floating-point `float`，**`decimal.Decimal` 更適合**，因為它可以用十進位方式表示並依需求設定舍入規則。

例如：

```python
from decimal import Decimal, ROUND_HALF_UP

price = Decimal("0.10")
tax = Decimal("0.20")
total = price + tax

print(total)
```

**結論：** 科學計算通常接受浮點近似並使用適當的誤差比較；金融等要求十進位精確度的情境，應考慮 `Decimal` 等十進位數值工具。不能單純靠 `round()` 解決所有浮點數問題。

## 最後整理

1. `0.1` 的二進位表示是 `0.00011001100110011...₂`，屬於無限循環小數。
2. Python 的 `float` 使用 IEEE 754 雙精度浮點數，無法精確保存所有十進位小數。
3. `sys.float_info` 可以查看 Python 浮點數的範圍與精度資訊。
4. `0.1 + 0.2` 出現 `0.30000000000000004` 是浮點數表示誤差造成的；比較時可使用 `math.isclose()`，需要十進位精確計算時可使用 `Decimal`。
5. 科學計算與金融計算的處理方式不同，應依精度需求選擇 `float`、容許誤差比較或 `Decimal`。